# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Define the dataset URL from Croissant schema
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields, referencing each by their `@id`.

First, let's inspect all available record sets in this dataset:

In [ ]:
# List all available record sets and their @id, @name and contained fields
record_sets = [rs for rs in dataset.metadata.record_sets]
for rs in record_sets:
    print(f"Record set: @id={rs.id}, name={rs.name}")
    print("  Fields:")
    for field in rs.fields:
        print(f"   - Field: @id={field.id}, name={field.name}, dataType={field.data_type}")
    print()

Let's print some sample records from one of the main record sets. (Replace with an actual record set `@id` as shown above for exploration):

In [ ]:
# Choose the primary record set to explore. According to FAIR2 Croissant schemas, the main table is typically the first record set.
# We extract its @id for further exploration.
if record_sets:
    main_rs = record_sets[0]  # You can change this if needed based on output above
    main_rs_id = main_rs.id
    print(f"Exploring first record set: @id={main_rs_id}, name={main_rs.name}")
    print("Sample records:")
    for idx, rec in enumerate(dataset.records(record_set=main_rs_id)):
        print(rec)
        if idx >= 2:
            break
else:
    print("No record sets found in dataset.")

## 3. Data Extraction
Load all data from each record set into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Show columns of the main record set DataFrame
if record_set_ids:
    print(f"Columns in record set (@id={main_rs_id}):\n{dataframes[main_rs_id].columns.tolist()}")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalizing, and grouping on a numeric field, referencing all fields by their `@id`.

In [ ]:
# Select a numeric field for analysis based on above showed columns; 
# For mass spec human EV proteomics, typical numeric fields are e.g. "MW" (molecular weight), "Coverage", "Abundance", "Peptides", etc.
# Please check the list of columns printed earlier to identify a suitable field.

import numpy as np

# Example: Let's try to use the first float/integer typed field for demonstration
numeric_field_id = None
for field in main_rs.fields:
    # common types: 'Float', 'Integer', 'Number'
    if 'Float' in field.data_type or 'Integer' in field.data_type or 'Number' in field.data_type:
        if field.id in dataframes[main_rs_id].columns:
            numeric_field_id = field.id
            print(f"Using numeric field: @id={numeric_field_id}, name={field.name}")
            break

if numeric_field_id is None:
    print("No numeric field found for demonstration.")
else:
    # Try to filter for values > threshold (e.g., MW > 10000)
    threshold = np.nanmean(dataframes[main_rs_id][numeric_field_id])
    filtered_df = dataframes[main_rs_id][dataframes[main_rs_id][numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} rows")
    display(filtered_df.head())

    # Normalize this column
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} (z-score):")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical/text field if available
    group_field_id = None
    for field in main_rs.fields:
        # Try to find a 'Text' type field that's non-unique
        if 'Text' in field.data_type and field.id in dataframes[main_rs_id].columns:
            uniq_vals = dataframes[main_rs_id][field.id].nunique()
            if uniq_vals > 1 and uniq_vals < len(dataframes[main_rs_id]) // 2:
                group_field_id = field.id
                print(f"Grouping by: @id={group_field_id}, name={field.name}")
                break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}: (showing top 5)")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the selected numeric field, if one was found
if numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[main_rs_id][numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        # Boxplot by group
        plt.figure(figsize=(10, 4))
        sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load, inspect, analyze, and visualize the FAIR² dataset on Mass Spectrometry Analysis of Extracellular Vesicles from Stimulated Human Mast Cells. By referencing all entities by their `@id` and using transparent, reproducible data access and manipulation steps, we ensured that any result here can be traced back to the dataset's Croissant schema definition. For further downstream analysis, see the record set, field, and column `@id`s above.